# Bonus: How to let the model decide when to search

**CPU | API key: `OPENAI_API_KEY` (chat and embedding calls) | Self-study**

## Goal

Give the model a search tool, let it decide whether and how often to search, and judge from answers, model calls and
tokens when that beats the pipeline that searches once with the question. The moment that teaches it: a question with
two parts gets one half wrong from the fixed pipeline, because the passage for that half ranks fourth and is never
read; the model splits it into two searches and answers both parts. On a simple question it costs an extra model
call, and on a greeting it skips a search the pipeline always makes.

The pipeline you have built searches once, with the question as typed or rewritten (Lab 09), and always reads
the top three passages. This particular baseline always makes one search; it has no query-decomposition or small-talk branch. A fixed
workflow could include those branches too. Here we give those decisions to the model and measure the complete design. That design is called agentic
RAG, against the two-step RAG of the fixed pipeline.

## What you will learn

- How a fixed search-then-answer pipeline handles a question with two parts, and a message that asks nothing.
- How to give the model a search tool and run its calls by hand, two calls in one reply included.
- How to add a stopping rule to the loop, and what a cap set too low costs.
- How to build the same loop with LangChain's `create_agent`, and read the graph it returns.
- How to compare two designs on the same questions in answers, model calls, searches and tokens, and decide which
  kind of question each one suits.

## Set up the notebook

Run these two cells first, every time; Lab 00 explains each line. **Setup** finds the repository folder (on Colab it
copies the repository and installs the packages first). **Config** reads the settings and finds your key.

In [ ]:
# Setup: on Colab, copy the repository and install the packages. Everywhere else they are installed already.
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/rag_for_nlp").exists():
        !git clone -q --depth 1 https://github.com/tatwan/rag_for_nlp.git /content/rag_for_nlp
    %cd /content/rag_for_nlp
    %pip install -q -r requirements.txt

ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "models.env").exists())
sys.path.insert(0, str(ROOT))
print("Repository folder:", ROOT.name)

In [ ]:
# Config: settings from models.env (your .env and environment variables win); the key from the environment, .env or Colab Secrets.
from shared.settings import load_settings, load_key, stamp

settings = load_settings()
key_found_in = load_key("OPENAI_API_KEY")

for name, value in stamp(settings).items():
    print(f"{name:18}{value}")
print(f"{'OPENAI_API_KEY':18}{'found in ' + key_found_in if key_found_in else 'not found'}")

The settings above identify the chat and embedding endpoints and models; `JSON_MODE` selects the fixed pipeline's
answer format. The corpus is embedded once, and every executed search embeds its query. Re-running cells makes
new calls. Part E reports chat tokens, including the growing tool conversation; it does not report embedding
usage, latency or a total bill. Charges depend on your provider and model.

The saved-output checkpoints describe the notebook's own run. Model replies and similarity scores vary, so inspect
the facts and ranking before comparing totals.

Two designs answer the same questions. The **fixed pipeline** searches once with the question and returns structured
claims with quotes. The **agent** gets a search tool and returns text with source IDs. Their prompts and answer
formats differ too: this compares the complete designs, rather than isolating retrieval alone.

---

## Part A: Run the fixed pipeline on three kinds of question

Part A shows what the fixed pipeline does with three kinds of message, so you know what the agent has to beat.

Build Lab 05's vector store with `shared/lc_pipeline.py`: the 272 chunks and the five policies, embedded once.

In [ ]:
from shared import lc_pipeline

store = lc_pipeline.build_store(lc_pipeline.to_documents(lc_pipeline.load_corpus()), settings)
print(len(store.store), "documents in the store")

`fixed()` is the whole pipeline in three lines: search with the question (k=3), write the hits as `[ID] text` evidence
lines, and ask for a cited answer with Lab 02's rules. `show()` prints what was searched, the answer, and Lab 02's
quote check on every claim. Start with a question that has one fact:

In [ ]:
from shared.answer import quote_found

def fixed(question):
    hits = lc_pipeline.retrieve(store, question)
    lines = lc_pipeline.evidence(hits)
    return hits, lines, lc_pipeline.answer(question, lines, settings)

def show(question):
    hits, lines, reply = fixed(question)
    print("passages:", [document.id for document, _ in hits])
    print("question:", question)
    print("answer:  ", reply.answer)
    for claim in reply.claims:
        print(f"  claim: {claim.claim}\n  [{claim.source_id}] quote found {quote_found(claim.model_dump(), lines)}: {claim.quote}")
    if "Normandy" in question:
        print("Evidence read by the answer model:\n" + "\n".join(lines))

show("Who was the Super Bowl 50 MVP?")

One search, three passages, and *Von Miller was the Super Bowl 50 MVP.*, its quote found in S00-P02-C00. One
question, one fact, one search: the case the pipeline is built for.

Now a question with two parts, both answered in the corpus: Richard I ruled the duchy, and the Latin form of *Norman*
was first recorded in the 9th century. Run it three times:

In [ ]:
NORMANDY = "Who ruled the duchy of Normandy, and when was the Latin version of the word Norman first recorded?"
for _ in range(3):
    show(NORMANDY)

Read the complete Normandy replies and their evidence. The ruler is Richard I. The Latin-form date is the 9th
century; *the 10th and 11th centuries* in S02-P00-C00 describes when the Normans gave their name to Normandy.
An answer that uses those centuries for the Latin word links real words to the wrong relationship. A quote can be
found in the evidence while the claim it is offered to support is wrong.

Where is the passage that holds the Latin date? Rank ten passages for the question and print that one:

In [ ]:
ranked = [document.id for document, _ in lc_pipeline.retrieve(store, NORMANDY, k=10)]
print(ranked)
print("S02-P02-C00 ranks", ranked.index("S02-P02-C00") + 1)
print(store.get_by_ids(["S02-P02-C00"])[0].page_content)

S02-P02-C00 says *recorded in Medieval Latin, 9th century*. When it ranks fourth, the fixed design
leaves it outside the top three. Read whether the model fills that gap from another passage mentioning Latin
and centuries, or recognises that its evidence lacks the date. This baseline searches the whole question once, so it never tries a separate query for the missing fact.
A fixed workflow with query decomposition could try that; Part B lets the model choose the queries.

The third kind is a message that asks nothing. A help desk gets greetings and thanks as well as questions:

In [ ]:
show("Hi there!")

Read the greeting's retrieved passages and reply. This fixed design makes one search and one model invocation
for every message. Its rules allow an answer from evidence or a refusal, so a greeting can receive an unnecessary
*Insufficient evidence.* reply.

### Checkpoint A

| Check | What to inspect |
|---|---|
| Store | 277 documents |
| Super Bowl | Von Miller, with the answer's full claim and quote checked against evidence |
| Normandy | Three complete replies: separate the ruler, the Latin date and the relationship the cited quote supports |
| Answering passage | S02-P02-C00; read its position and the 9th-century statement |
| Greeting | A search executes even though no document fact was requested |

---

## Part B: Give the model a search tool and run its calls by hand

Part B makes the model's search request visible before anything executes. `@tool` exposes the function's name,
argument schema and docstring. `bind_tools()` sends that description with the prompt; it does not run the function.
An assistant reply may contain several requested calls, each recorded as a dictionary.

Start with the Normandy question. Print every complete call record, then inspect one returned `ToolMessage`
before putting the round trip in a loop.

In [ ]:
from langchain.tools import tool
from shared.settings import lc_chat_model, chat_url

@tool
def search_documents(query: str) -> str:
    """Search the documents and return the three closest passages as [ID] text lines."""
    return "\n".join(lc_pipeline.evidence(lc_pipeline.retrieve(store, query)))

SYSTEM = ("Answer questions from the documents. Use search_documents to find passages; you may search more than once."
          " Answer only from the passages and cite their IDs in brackets."
          " If the passages do not answer, say: Insufficient evidence.")
chat = lc_chat_model("CHAT_MODEL", settings)
with_search = chat.bind_tools([search_documents])
messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": NORMANDY}]
reply = with_search.invoke(messages)
print("text:", repr(reply.content))
for call in reply.tool_calls:
    print("tool call:", call)

The assistant can request two searches in one reply. The API calls these **parallel tool calls**: they are proposed
together. Our Python `for` loop will execute them one at a time.

Each record has `name` (the function), `args` (its query), `id` (this request's identifier) and `type` (`tool_call`).
Nothing has searched yet. Append the assistant reply first, then execute its first call and inspect the result.
The result's `tool_call_id` must match that call's `id`, so the next model call can associate evidence with its request.

In [ ]:
import re

messages.append(reply)
call = reply.tool_calls[0]
result = search_documents.invoke(call)
messages.append(result)
print(result.model_dump())
print("IDs match:", result.tool_call_id == call["id"])

The `ToolMessage` stores the returned passages in `content`, the function in `name`, and the matching request in
`tool_call_id`. Its `status` is `success` for an executed search. This says the tool ran, not that its passages answer
the question.

Complete the other requested calls before invoking the model again. Sending only one result for a two-call reply
leaves an unmatched request and the API rejects the conversation. The last reply below is printed in full; if it
requests more tools, the next step generalises these round trips into a function for new questions.

In [ ]:
for call in reply.tool_calls[1:]:
    result = search_documents.invoke(call)
    messages.append(result)
    print(f"{call['args']['query']!r} found", re.findall(r"^\[(.+?)\]", result.content, re.M))

final = with_search.invoke(messages)
print("tool calls:", final.tool_calls)
print(final.content)

Read the second search's IDs and the final answer together. A separate query can bring S02-P02-C00 into the top
three, supplying the Latin date that the whole-question search missed. A model may instead request another search
here, so one send-back is not a complete agent.

Put the round trip in a loop: append the assistant reply, execute each request, append every result, then ask again.
It stops when a reply has no tool calls. The `for` loop also allows at most four model invocations;
if none supplies an answer, Python returns `STOP`. This is a model-call limit, not a search budget: one reply
can request several searches. Part C limits those searches separately. `queries` records executed searches and `calls` counts model invocations.

In [ ]:
STOP = "Model-call limit reached. No final answer was produced."

def run_agent(question, max_model_calls=4):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    queries = []
    for calls in range(1, max_model_calls + 1):
        reply = with_search.invoke(messages)
        if not reply.tool_calls:
            return queries, calls, reply.content
        messages.append(reply)
        for call in reply.tool_calls:
            queries.append(call["args"]["query"])
            messages.append(search_documents.invoke(call))
    return queries, max_model_calls, STOP

for question in ("Who was the Super Bowl 50 MVP?", "Hi there!"):
    print(*run_agent(question), sep=" | ")

Compare the two query lists. The fixed pipeline always searches; the agent may skip search for a greeting.
A factual request normally needs a request round and an answer round, adding a model invocation to the baseline.
Read the full replies too: skipping search alone does not establish a useful or supported answer.

### Checkpoint B

| Check | What to inspect |
|---|---|
| Normandy request | Complete call records, with separate IDs for separate requested searches |
| First tool response | Full `ToolMessage`; `tool_call_id` equals the requested call's `id` |
| After the tool results | The complete model reply and any further requests |
| Other messages | Executed-query lists, model-call counts and complete replies for Super Bowl and the greeting |

---

## Part C: Add a stopping rule

Part C separates two limits: how many searches may execute and how many times the model may be called. The first
loop bounds model calls but runs every requested search. First observe its behaviour on a question the documents
do not cover, then add a search budget.

In [ ]:
REFUND = "What is the refund policy for Boreal?"
for _ in range(3):
    print(*run_agent(REFUND), sep=" | ")

Read the refund replies above: a model can search again before deciding the documents do not answer.

A **search budget** allows at most `max_searches` actual searches. Later requests get a `ToolMessage` containing
`LIMIT`, so every request still has a matching result. That message asks the model to answer; it cannot force the
model to stop requesting tools.

A separate **hard model-call limit** allows at most `max_model_calls` invocations of `with_search.invoke()`.
The `for` loop enforces it even if every reply requests another search. If it runs out, Python returns `STOP`
instead of making another model call. This terminal message is not an evidence-based answer. Four model calls and
three searches are small teaching budgets; they are not provider limits or a time limit for one stalled request.

In [ ]:
from langchain_core.messages import ToolMessage

LIMIT = "Search limit reached. Answer from the passages you already have."
STOP = "Model-call limit reached. No final answer was produced."

def run_agent(question, max_searches=3, max_model_calls=4):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    queries = []
    for calls in range(1, max_model_calls + 1):
        reply = with_search.invoke(messages)
        if not reply.tool_calls:
            return queries, calls, reply.content
        messages.append(reply)
        for call in reply.tool_calls:
            if len(queries) < max_searches:
                queries.append(call["args"]["query"])
                messages.append(search_documents.invoke(call))
            else:
                messages.append(ToolMessage(LIMIT, tool_call_id=call["id"], status="error"))
    return queries, max_model_calls, STOP

A limit has to be tested where it bites. Set it to one search, and run the refund question and the Normandy question
three times each:

In [ ]:
for question in (REFUND, NORMANDY):
    for _ in range(3):
        print(*run_agent(question, max_searches=1), sep=" | ")

Compare each full reply with its executed-query list. With one search allowed, later requests receive a limit
message; the model may then answer incompletely, refuse, or request more tools until the hard limit ends the run.
A budget can remove evidence a correct answer needs. Set limits from measured questions and retain a clear terminal
outcome when the budget is exhausted.

### Checkpoint C

| Check | What it establishes |
|---|---|
| Refund replies | Which queries actually executed before each refusal or terminal message |
| One-search runs | At most one query in each executed-query list; read whether the Normandy date survived |
| Hard limit | At most four model invocations, even if requests continue; `STOP` means no final model answer |

---

## Part D: Build the same loop with `create_agent`

Part D hands the loop to LangChain. [`create_agent`](https://docs.langchain.com/oss/python/langchain/agents) takes
the model, the tools and the system prompt, and runs the steps `run_agent()` runs. Build it and display what it
returns:

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware

agent = create_agent(chat, tools=[search_documents], system_prompt=SYSTEM,
                     middleware=[ModelCallLimitMiddleware(run_limit=4, exit_behavior="end")])
print(type(agent).__name__)
print(agent.get_graph().draw_mermaid())

`create_agent` returns a LangGraph `CompiledStateGraph`. Its Mermaid text lists the compiled nodes and edges.
The saved picture below explains the loop and the model-call guard without a rendering service.

![Agent loop: check the model-call limit, call the model, execute requested searches, and return results.](resources/agent-loop.png)

Read arrows as the next execution step. A model reply with tool calls goes to `tools`; a reply without them ends.
Tool results return through the call-limit check before another model invocation. At four calls the guard ends
with a generated limit notice, without asking the model again. The graph adds middleware hooks around the same
message round trip we wrote by hand. A recursion limit counts graph steps and is separate from these model calls.

Run the Normandy question. `show_messages()` prints requests, tool results and the final reply in full.

In [ ]:
def show_messages(messages):
    for message in messages:
        print(f"{message.type} |", getattr(message, "tool_calls", []) or message.text)

show_messages(agent.invoke({"messages": [{"role": "user", "content": NORMANDY}]})["messages"])

Follow the assistant requests and their returned evidence to the final answer. Now add a **middleware**, a hook
around the model or tools, for the search budget too. The tool limit's `continue` behaviour returns error tool
messages for blocked requests. It leaves the model free to ask again; the separate model-call guard bounds that
continuation. Set one search and four model calls so we can see what is blocked.

In [ ]:
from langchain.agents.middleware import ToolCallLimitMiddleware

tight = create_agent(chat, tools=[search_documents], system_prompt=SYSTEM,
                     middleware=[ModelCallLimitMiddleware(run_limit=4, exit_behavior="end"),
                                 ToolCallLimitMiddleware(run_limit=1, exit_behavior="continue")])
tight_result = tight.invoke({"messages": [{"role": "user", "content": NORMANDY}]})
show_messages(tight_result["messages"])
print("Model calls:", sum(m.type == "ai" and m.usage_metadata is not None for m in tight_result["messages"]))
print("Executed searches:", sum(m.type == "tool" and m.status == "success" for m in tight_result["messages"]))
print("Blocked requests:", sum(m.type == "tool" and m.status == "error" for m in tight_result["messages"]))

Read every blocked request above, including repeated requests. The count is per request, not per distinct query.
The successful tool message identifies the search that actually ran; error messages identify attempts that did not.
If the model still requests tools after its fourth call, the guard returns a limit notice, which is an application
message rather than another model reply.

For Part E allow three searches and four model calls. Print the nodes to see both guards in the compiled graph.

In [ ]:
agent = create_agent(chat, tools=[search_documents], system_prompt=SYSTEM,
                     middleware=[ModelCallLimitMiddleware(run_limit=4, exit_behavior="end"),
                                 ToolCallLimitMiddleware(run_limit=3, exit_behavior="continue")])
print(list(agent.get_graph().nodes))

### Checkpoint D

| Check | What to read |
|---|---|
| Graph | Model and tool nodes plus middleware hooks; the saved picture explains the decision edges |
| Normandy | Requested queries, their full tool evidence and the complete final reply |
| One-search budget | One executed search at most; count all blocked requests, including repeats, from this run |
| Hard call limit | At most four provider AI replies with usage; the generated limit notice is not a model call |

---

## Part E: Compare the two designs on the same questions

Part E compares the complete designs on eight constructed test messages, three runs each. Some questions reuse
public SQuAD facts; Atlas and Boreal are constructed policies. These are development examples, not the protected
question set.

Each tuple stores `kind` (the reporting group), `question` (sent to the design) and `parts` (the evaluator's answer
key). Each inner list contains accepted phrasings for one required fact. Normandy has two parts, one for the ruler
and one for the date. The key and group label are never passed to the answering model. It sees its own system/tool
instructions, the question and any retrieved evidence.

In [ ]:
QUESTIONS = [
    ("one fact", "Who was the Super Bowl 50 MVP?", [["Von Miller"]]),
    ("one fact", "In what year was Nikola Tesla born?", [["1856"]]),
    ("one fact", "Who can export all reports in Atlas?", [["workspace owners"]]),
    ("two parts", NORMANDY, [["Richard I"], ["9th century"]]),
    ("two parts", "Who was the Super Bowl 50 MVP, and who can export all reports in Atlas?",
     [["Von Miller"], ["workspace owners"]]),
    ("two parts", "How long is an Atlas reset link valid, and how long is a Boreal one?", [["30 minutes"], ["10 minutes"]]),
    ("no search", "Hi there!", []),
    ("no search", "Thanks, that is all I needed.", []),
]
print(len(QUESTIONS), "messages:", sorted({kind for kind, _, _ in QUESTIONS}))

We keep one result dictionary for each question, design and repetition:

- **Accepted:** `all(found(answer, part) for part in parts)` requires a matching accepted phrase for every part.
  `found()` normalises text and matches whole words. It cannot check relationships, negation or support. Reversing
  the Atlas and Boreal durations still contains both accepted phrases. For empty `parts`, we instead test that the
  exact `Insufficient evidence.` string is absent; that alone does not establish a useful small-talk reply.
- **Model calls:** one invocation in the fixed design. For the agent we count AI messages carrying provider `usage_metadata`;
  the synthetic limit notice has none. An endpoint omitting usage would make this count incomplete. These are logical calls; transport retries are not separately counted.
- **Attempted / executed searches:** every requested tool call counts as an attempt, even when blocked. A successful
  `ToolMessage` counts an executed search in this lab; a middleware error message counts a blocked attempt.
- **Chat tokens:** the callback sums input and output usage across model invocations, including the fixed design's
  structured answer. Query embeddings and corpus embeddings are outside this callback.

First define the wrappers and scorer. Then carry Normandy through one real run before the larger sweep.

In [ ]:
from langchain_core.callbacks import get_usage_metadata_callback
from shared.evals import REFUSAL, found

def fixed_design(question):
    hits, lines, reply = fixed(question)
    return reply.answer, [question], 1, 1

def agent_design(question):
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    messages = result["messages"]
    queries = [call["args"]["query"] for message in messages if message.type == "ai" for call in message.tool_calls]
    executed = sum(message.type == "tool" and message.status == "success" for message in messages)
    return messages[-1].text, queries, sum(m.type == "ai" and m.usage_metadata is not None for m in messages), executed

def run(design, question, parts):
    with get_usage_metadata_callback() as usage:
        answer, queries, calls, executed = design(question)
    tokens = [sum(u[kind] for u in usage.usage_metadata.values()) for kind in ("input_tokens", "output_tokens")]
    accepted = all(found(answer, part) for part in parts) if parts else REFUSAL not in answer
    return {"accepted": accepted, "calls": calls, "attempted": len(queries), "searches": executed,
            "tokens": tokens, "queries": queries, "answer": answer}

In [ ]:
kind, question, parts = QUESTIONS[3]
print({"kind": kind, "question": question, "parts": parts})
one = run(agent_design, question, parts)
print(one)

The printed record ties Normandy's question to its two required phrases. The returned row keeps its answer,
requested queries, executed-search count, calls and chat-token pair (`[input, output]`). `accepted=True` says both
phrases were found, not that the citations support the answer.

Repeat each existing tuple three times through each design. Store the question in its row, so labels cannot be
separated from their answer when we filter or group results. Print complete replies so a missing fact or changed
qualification remains readable in the saved notebook.

In [ ]:
rows = []
for kind, question, parts in QUESTIONS:
    print("\nQuestion:", question)
    for name, design in (("fixed", fixed_design), ("agent", agent_design)):
        for repeat in range(1, 4):
            row = {"kind": kind, "question": question, "design": name, "repeat": repeat,
                   **run(design, question, parts)}
            rows.append(row)
            print(f"{name} run {repeat} | accepted={row['accepted']} | {row['calls']} model calls | "
                  f"{row['attempted']} attempted / {row['searches']} executed searches | chat tokens {row['tokens']}")
            print("Requested queries:", row["queries"])
            print(row["answer"])

Read the full failed replies as well as the accepted ones. For example, does the mixed Super Bowl/Atlas question
lose a fact, assign it to the wrong subject, or refuse? Inspect the thank-you too: a reply can skip search and
still refuse unnecessarily.

Now filter `rows` by `kind` and `design`. Each `group` contains the three repetitions of that group's existing
questions. Sum booleans for accepted counts and sum numeric fields for work and tokens. These are repeated runs
on eight messages, not 24 independent questions.

In [ ]:
print(f"{'kind':11}{'design':8}{'accepted':>10}{'calls':>8}{'attempts':>10}{'executed':>10}{'chat in':>10}{'chat out':>10}")
for kind in ("one fact", "two parts", "no search"):
    for name in ("fixed", "agent"):
        group = [row for row in rows if row["kind"] == kind and row["design"] == name]
        print(f"{kind:11}{name:8}{sum(row['accepted'] for row in group):>4} of {len(group)}"
              f"{sum(row['calls'] for row in group):>8}{sum(row['attempted'] for row in group):>10}"
              f"{sum(row['searches'] for row in group):>10}{sum(row['tokens'][0] for row in group):>10}"
              f"{sum(row['tokens'][1] for row in group):>10}")
print(f"\n{settings['CHAT_MODEL']} at {chat_url(settings)}")

Compare each kind separately. A one-fact question can cost the agent a request round plus an answer round where
the fixed design needs one model call. For a compound question, inspect which facts the extra queries recovered.
For greetings and thanks, distinguish skipping search from giving a useful reply.

The chat-token totals also reflect different prompts and answer formats: the fixed design writes claims with quotes;
the agent writes text with IDs. The agent has no quoted claims for `quote_found()` to check. Before adopting it,
read support and citations as well as accepted phrases. No latency was measured, and the callback excludes embedding
cost, so the table alone cannot establish which design is faster or cheaper overall.

### Checkpoint E

| Check | What it permits |
|---|---|
| Accepted counts | Report phrase matches by kind, then inspect the complete answers for factual support |
| Calls and searches | Distinguish requested calls from executed searches and model invocations |
| Repetitions | Observe stability on these eight messages; no claim of broad question coverage |
| Chat tokens | Compare this usage component for the two complete designs |

---

## What you can do now

- Show where a fixed pipeline fails on a two-part question while every citation check passes, and what it does with
  a greeting.
- Give the model a search tool and run its calls by hand, two in one reply included, in a loop.
- Separate a search budget from a hard model-call limit, and test both.
- Build the same agent with `create_agent`, `ToolCallLimitMiddleware` and `ModelCallLimitMiddleware`, and read its graph.
- Compare two designs in answers, model calls, searches and tokens, and say which kind of question each suits.

## Challenge

Use only what this lab showed. Task 1 is the challenge; Task 2 is optional. Keep the complete replies and your
answer to **Explain**. These questions can expose a limit or regression; the outcome depends on the model.

1. **Ask a question with three parts.** *Who was the Super Bowl 50 MVP, in what year was Nikola Tesla born, and who can
   export all reports in Atlas?*

   Set `THREE_PARTS` to the question and `PARTS` to one accepted-answer list per fact. Run both designs three times.

   **Check:** inspect the fixed pipeline's passages, every full answer, accepted phrases, attempted/executed searches
   and chat tokens. A phrase pass still needs evidence inspection.

   **Explain:** which fact, if any, did the pipeline miss? How many searches did the agent use? Would another fact
   necessarily need another search? Propose what you would test before adding a fourth part: one query can retrieve
   passages containing several required facts, and the model may combine or repeat queries.

2. **Optional: let the agent answer small talk.** Add one line to `SYSTEM`, such as *Reply to greetings and thanks in
   one short sentence, without searching.* Test the no-search messages and all three ordinary one-fact questions,
   three times each.

   **Check:** print full replies and requested/executed searches. Check that ordinary factual answers still use
   document evidence and cite IDs, even when their accepted phrases appear.

   **Explain:** which rows did the line improve or harm? Why would testing greetings and thanks alone miss a
   regression in ordinary questions?

**Task 1.** Your question and its parts, then both designs:

In [ ]:
THREE_PARTS = None    # TODO: the question from the task, as a string
PARTS = None          # TODO: one list of accepted answers per part, as in QUESTIONS

if THREE_PARTS is None or PARTS is None:
    print("⚠️ Set THREE_PARTS and PARTS, then run the cell again.")
else:
    print("pipeline passages:", [document.id for document, _ in lc_pipeline.retrieve(store, THREE_PARTS)])
    for name, design in (("fixed", fixed_design), ("agent", agent_design)):
        for _ in range(3):
            row = run(design, THREE_PARTS, PARTS)
            print(name, {key: value for key, value in row.items() if key != "answer"})
            print(row["answer"])

**Task 2 (optional).** Your line, then the five messages:

In [ ]:
SMALL_TALK = None    # TODO: one line to add to SYSTEM, as a string

if SMALL_TALK is None:
    print("⚠️ Set SMALL_TALK, then run the cell again.")
else:
    polite = create_agent(chat, tools=[search_documents], system_prompt=SYSTEM + " " + SMALL_TALK,
                          middleware=[ModelCallLimitMiddleware(run_limit=4, exit_behavior="end"),
                                      ToolCallLimitMiddleware(run_limit=3, exit_behavior="continue")])
    for kind, question, parts in QUESTIONS:
        if kind in ("no search", "one fact"):
            for _ in range(3):
                result = polite.invoke({"messages": [{"role": "user", "content": question}]})
                messages = result["messages"]
                queries = [call["args"]["query"] for message in messages if message.type == "ai" for call in message.tool_calls]
                executed = sum(message.type == "tool" and message.status == "success" for message in messages)
                print("Question:", question)
                print("Requested queries:", queries, "| executed searches:", executed)
                print(messages[-1].text)

## Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `OpenAIInvalidRequestError: Error code: 400 ... An assistant message with 'tool_calls' must be followed by tool messages responding to each 'tool_call_id'.` | a reply with two tool calls was sent back with a result for only one of them | Append a `ToolMessage` for every call in `reply.tool_calls`, a limit message included, before the next `invoke()` |
| `GraphRecursionError: Recursion limit of 2 reached without hitting a stop condition.` | a `{"recursion_limit": 2}` in the `invoke` config: it counts graph steps, and one search round takes two | Use separate tool and model-call limits as in Part D; keep the recursion limit above the required graph steps |
| `ValueError: Failed to reach https://mermaid.ink API while trying to render your graph` when you display `agent` | the picture is drawn by the mermaid.ink web service, which your network blocks | Use the bundled `resources/agent-loop.png` picture and `print(agent.get_graph().draw_mermaid())` for the compiled nodes; neither needs that service |
| `OpenAIInvalidRequestError: Error code: 400 ... Function tools with reasoning_effort are not supported for gpt-6-luna in /v1/chat/completions.` | `REASONING_EFFORT` is `low`, `medium` or `high` for a reasoning model | Set `REASONING_EFFORT=none` in `.env`: Chat Completions takes tools on `gpt-6-luna` only at that effort |

## Resources

- [LangChain: retrieval](https://docs.langchain.com/oss/python/langchain/retrieval): 2-step RAG against agentic RAG,
  where the model decides when to retrieve
- [LangChain: agents](https://docs.langchain.com/oss/python/langchain/agents): `create_agent`, a model calling tools
  in a loop, and the middleware around it
- [LangChain: built-in middleware](https://docs.langchain.com/oss/python/langchain/middleware/built-in): the tool
  call limit and the model call limit
- [LangChain: models, token usage](https://docs.langchain.com/oss/python/langchain/models#token-usage):
  `usage_metadata` and `get_usage_metadata_callback()`
- [OpenAI: function calling](https://developers.openai.com/api/docs/guides/function-calling): tool calls, parallel
  calls in one reply, and the tool results the API expects back
- [Asai et al., *Self-RAG*, ICLR 2024](https://arxiv.org/abs/2310.11511): a model trained to decide when to retrieve

Back to [the lab list](../README.md).